In [1]:
import pandas as pd
import numpy as nm
import os

In [2]:
from pathlib import Path

In [3]:
DATA_DIR = Path("../dataset")

CHECKING MISSING COUNTS

In [4]:
for file in DATA_DIR.glob("*.csv"):
    filename = file.name

    filepath = DATA_DIR/filename

    if not filepath.exists():
        print("missing file", filename)
        continue

    print(f"\nChecking : {filename}") 

    missing_counts = None
    dtypes = None
    total_rows = 0

    for chunk in pd.read_csv(filepath, chunksize=200_000):

        chunk_missing = chunk.isna().sum()

        if missing_counts is None:
            missing_counts = chunk_missing
            dtypes = chunk.dtypes.astype(str)
        else:
            missing_counts += chunk_missing

        total_rows += len(chunk)


    summury = pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percentage": (missing_counts / total_rows * 100).round(2),
        "data_type": dtypes
    })

    print("Total rows :", total_rows)
    display(summury)




Checking : aisles.csv
Total rows : 134


,missing_count,missing_percentage,data_type
aisle_id,0,0.0,int64
aisle,0,0.0,str



Checking : departments.csv
Total rows : 21


,missing_count,missing_percentage,data_type
department_id,0,0.0,int64
department,0,0.0,str



Checking : orders.csv
Total rows : 3421083


,missing_count,missing_percentage,data_type
order_id,0,0.00,int64
user_id,0,0.00,int64
eval_set,0,0.00,str
order_number,0,0.00,int64
order_dow,0,0.00,int64
order_hour_of_day,0,0.00,int64
days_since_prior_order,206209,6.03,float64


... (2 outputs left out)


Checking : order_products__train.csv
Total rows : 1384617


,missing_count,missing_percentage,data_type
order_id,0,0.0,int64
product_id,0,0.0,int64
add_to_cart_order,0,0.0,int64
reordered,0,0.0,int64



Checking : products.csv
Total rows : 49688


,missing_count,missing_percentage,data_type
product_id,0,0.0,int64
product_name,0,0.0,str
aisle_id,0,0.0,int64
department_id,0,0.0,int64


CHEKING DUPLICATES RECORDS

##### checking for small files


In [5]:
small_files = {

    "orders.csv": "order_id",
    "products.csv": "product_id",
    "aisles.csv": "aisle_id",
    "departments.csv": "department_id",
}

for filename, key in small_files.items():

    df = pd.read_csv(DATA_DIR/filename)

    duplicate_count = df.duplicated(subset = [key]).sum()

    print(f"{filename}: {duplicate_count} duplicate keys")




orders.csv: 0 duplicate keys
products.csv: 0 duplicate keys
aisles.csv: 0 duplicate keys
departments.csv: 0 duplicate keys


#### checking for large files


In [6]:
large_files = {
    "order_products__prior.csv",
    "order_products__train.csv"
}

for filename in large_files:
    duplicate_count = 0

    for chunk in pd.read_csv(DATA_DIR/filename, chunksize=200_000):
        duplicate_count += chunk.duplicated(subset=["order_id", "product_id"]).sum()

    print(f"{filename} : {duplicate_count} duplicate pairs withing chunks")


order_products__prior.csv : 0 duplicate pairs withing chunks
order_products__train.csv : 0 duplicate pairs withing chunks



CHECKING INVALID VALUES

#### checking for order.csv

In [7]:
orders = pd.read_csv(DATA_DIR/ "orders.csv")


print("Invalid order dow:",(~orders["order_dow"].between(0, 6)).sum())

print("Invalid order_hour_of_day :",(~orders["order_hour_of_day"].between(0, 23)).sum())

print("Invalid order_number:",(orders["order_number"]<= 0).sum())

print("Invalid user_id:",(orders["user_id"]<= 0).sum())

Invalid order dow: 0
Invalid order_hour_of_day : 0
Invalid order_number: 0
Invalid user_id: 0


##### checking for large files

In [8]:
large_files = {
    "order_products__prior.csv",
    "order_products__train.csv"
}

for filename in large_files:

    invalid_reordered = 0
    invalid_cart_order = 0
    invalid_ids = 0

    for chunk in pd.read_csv(DATA_DIR/filename, chunksize=200_000):
        invalid_reordered += (~chunk["reordered"].isin([0, 1])).sum()

        invalid_cart_order += (chunk["add_to_cart_order"]<= 0).sum()

        invalid_ids += (
            (chunk["order_id"] <= 0) | 
            (chunk["product_id"] <= 0)
        ).sum()

    print("\nFile : ", filename)
    print("Invalid reordered:", invalid_reordered)
    print("Invalid cart order:", invalid_cart_order)
    print("Invalid IDs:", invalid_ids)
    




File :  order_products__prior.csv
Invalid reordered: 0
Invalid cart order: 0
Invalid IDs: 0

File :  order_products__train.csv
Invalid reordered: 0
Invalid cart order: 0
Invalid IDs: 0


In [9]:
products = pd.read_csv(DATA_DIR / "products.csv")
aisles = pd.read_csv(DATA_DIR / "aisles.csv")
departments = pd.read_csv(DATA_DIR / "departments.csv")

print("Invalid product IDs:",
      (products["product_id"] <= 0).sum())

print("Invalid aisle IDs:",
      (products["aisle_id"] <= 0).sum())

print("Invalid department IDs:",
      (products["department_id"] <= 0).sum())

Invalid product IDs: 0
Invalid aisle IDs: 0
Invalid department IDs: 0


#### checking relationship between tables (referential integrity)

In [10]:
orders = pd.read_csv(DATA_DIR / "orders.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
aisles = pd.read_csv(DATA_DIR / "aisles.csv")
departments = pd.read_csv(DATA_DIR / "departments.csv")

In [11]:
invalid_aisles = products[
    ~products["aisle_id"].isin(aisles["aisle_id"])
]

invalid_departments = products[
    ~products["department_id"].isin(departments["department_id"])
]

print("Products with invalid aisle IDs:", len(invalid_aisles))
print("Products with invalid department IDs:", len(invalid_departments))

Products with invalid aisle IDs: 0
Products with invalid department IDs: 0


In [12]:
valid_order_ids = set(orders["order_id"])
valid_product_ids = set(products["product_id"])

large_files = [
    "order_products__prior.csv",
    "order_products__train.csv"
]

for filename in large_files:

    invalid_orders = 0
    invalid_products = 0

    for chunk in pd.read_csv(
        DATA_DIR / filename,
        chunksize=200_000
    ):

        invalid_orders += (
            ~chunk["order_id"].isin(valid_order_ids)
        ).sum()

        invalid_products += (
            ~chunk["product_id"].isin(valid_product_ids)
        ).sum()

    print("\nFile:", filename)
    print("Invalid order references:", invalid_orders)
    print("Invalid product references:", invalid_products)


File: order_products__prior.csv
Invalid order references: 0
Invalid product references: 0

File: order_products__train.csv
Invalid order references: 0
Invalid product references: 0


## DATA CLEANING

HANDLING DUPLICATE VALUE

0 duplicate count from all this dataset

HANDLING MISSING VALUE

We found only one column with missing values:

orders.csv → days_since_prior_order

All 206,209 missing values belong to first orders. These are legitimate missing values



DATA TYPE HANDLING

Our earlier analysis showed appropriate data types. Therefore, no conversions are currently required.

CREATING CLEANED DATASET FOLDER

In [13]:
CLEAN_DIR = Path("../outputs/cleaned_data")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Cleaned data folder:", CLEAN_DIR)
print("Folder exists:", CLEAN_DIR.exists())


Cleaned data folder: ..\outputs\cleaned_data
Folder exists: True




here no cleaning changes needed so far , we can can copy original files into this folder without modifying their records


In [14]:
import shutil

for file in DATA_DIR.glob("*.csv"):
    destination = CLEAN_DIR/file.name
    shutil.copy2(file, destination)

    print("Copied :", file.name)

Copied : aisles.csv
Copied : departments.csv
Copied : orders.csv
Copied : order_products__prior.csv
Copied : order_products__train.csv
Copied : products.csv


## Data Integration

Now we'll connect related tables using common columns.

here we are not combining all six csv files into one because of it will result into massive Dataframe

In [15]:
orders = pd.read_csv(CLEAN_DIR/"orders.csv")
products = pd.read_csv(CLEAN_DIR/"products.csv")

print("Orders shape", orders.shape)
print("products shape", products.shape)

print("\norders :")
display(orders.head())

print("\nproducts :")
display(products.head())

Orders shape (3421083, 7)
products shape (49688, 4)

orders :


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0



products :


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [16]:
order_products_sample = pd.read_csv(
    CLEAN_DIR / "order_products__prior.csv",
    nrows=10000
)

# join orders details
merged_data = pd.merge(
    orders,
    order_products_sample,
    on="order_id",
    how="inner"
)

# join products details
merged_data = pd.merge(
    merged_data,
    products,
    on="product_id",
    how="left"
)

display(merged_data.head(10))

print("merge shape", merged_data.shape)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id
0,40,382,prior,29,1,15,23.0,10070,1,1,Organic 1% Low Fat Milk,84,16
1,40,382,prior,29,1,15,23.0,42450,2,1,Macaroni & Cheese,38,1
2,40,382,prior,29,1,15,23.0,33198,3,1,Sparkling Natural Mineral Water,115,7
3,40,382,prior,29,1,15,23.0,34866,4,1,Chocolate Milk 1% Milkfat,84,16
4,214,503,prior,5,0,16,9.0,5499,1,0,Organic Cream Of Chicken Condensed Soup,69,15
5,214,503,prior,5,0,16,9.0,8424,2,0,Broccoli Crown,83,4
6,214,503,prior,5,0,16,9.0,37053,3,0,Just Mayo,72,13
7,214,503,prior,5,0,16,9.0,24852,4,1,Banana,24,4
8,214,503,prior,5,0,16,9.0,21329,5,0,Organic Hearty Split Pea & Uncured Ham Soup,69,15
9,214,503,prior,5,0,16,9.0,25093,6,1,Colby Jack Cheese,21,16


merge shape (10000, 13)


In [17]:
print("Merged shape:", merged_data.shape)

print("\nMissing values:")
print(merged_data.isnull().sum())

print("\nUnique orders:", merged_data["order_id"].nunique())

print("Unique customers:", merged_data["user_id"].nunique())

print("Unique products:", merged_data["product_id"].nunique())

Merged shape: (10000, 13)

Missing values:
order_id                    0
user_id                     0
eval_set                    0
order_number                0
order_dow                   0
order_hour_of_day           0
days_since_prior_order    504
product_id                  0
add_to_cart_order           0
reordered                   0
product_name                0
aisle_id                    0
department_id               0
dtype: int64

Unique orders: 977
Unique customers: 972
Unique products: 4511


to merge aisles and department we need to establish connection 

here product.csv contain both aisles_id and department_id

In [18]:
#samples
aisles = pd.read_csv(CLEAN_DIR / "aisles.csv")
departments = pd.read_csv(CLEAN_DIR / "departments.csv")

print("Aisles shape:", aisles.shape)
print("Departments shape:", departments.shape)

display(aisles.head())
display(departments.head())

Aisles shape: (134, 2)
Departments shape: (21, 2)


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [19]:
products_integrated = pd.merge(
    products,
    aisles,
    on="aisle_id",
    how="inner"
)

products_integrated = pd.merge(
    products_integrated,
    departments,
    on="department_id",
    how="left"
)

display(products_integrated.head(10))

print("Integreated shape: ", products_integrated.shape)

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry
5,6,Dry Nose Oil,11,11,cold flu allergy,personal care
6,7,Pure Coconut Water With Orange,98,7,juice nectars,beverages
7,8,Cut Russet Potatoes Steam N' Mash,116,1,frozen produce,frozen
8,9,Light Strawberry Blueberry Yogurt,120,16,yogurt,dairy eggs
9,10,Sparkling Orange Juice & Prickly Pear Beverage,115,7,water seltzer sparkling water,beverages


Integreated shape:  (49688, 6)


In [20]:
#verifying the integration
print("Missing aisle names:",
      products_integrated["aisle"].isnull().sum())

print("Missing department names:",
      products_integrated["department"].isnull().sum())

print("Unique products:",
      products_integrated["product_id"].nunique())

Missing aisle names: 0
Missing department names: 0
Unique products: 49688


In [21]:
print("Products:", products.shape[0])
print("Integrated products:", products_integrated.shape[0])

print("Unique product IDs:",
      products_integrated["product_id"].nunique())

print("Missing aisle names:",
      products_integrated["aisle"].isna().sum())

print("Missing department names:",
      products_integrated["department"].isna().sum())

Products: 49688
Integrated products: 49688
Unique product IDs: 49688
Missing aisle names: 0
Missing department names: 0


### Integrating the complete purchase history

previously 
performing on only 10000 records, not processing all 32.4 million records 

###### We'll use DuckDB because it can query CSV files directly, join large tables and save the result as Parquet without requiring us to load the entire dataset into a Pandas DataFrame

In [22]:
import duckdb

In [23]:
INTEGRATED_DIR = Path("../outputs/integrated_data")
INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)

print("Integrated data folder : ", INTEGRATED_DIR.resolve())

Integrated data folder :  C:\Users\KETAN SANJAY NIKAM\Desktop\mtech\dma\mini-project-nov\outputs\integrated_data


In [24]:
print(duckdb.__version__)

2.0.0


In [25]:
# Get absolute file paths
orders_path = (CLEAN_DIR / "orders.csv").resolve().as_posix()
prior_path = (CLEAN_DIR / "order_products__prior.csv").resolve().as_posix()
products_path = (CLEAN_DIR / "products.csv").resolve().as_posix()
aisles_path = (CLEAN_DIR / "aisles.csv").resolve().as_posix()
departments_path = (CLEAN_DIR / "departments.csv").resolve().as_posix()


print(orders_path)
print(prior_path)
print(products_path)
print(aisles_path)
print(departments_path)

C:/Users/KETAN SANJAY NIKAM/Desktop/mtech/dma/mini-project-nov/outputs/cleaned_data/orders.csv
C:/Users/KETAN SANJAY NIKAM/Desktop/mtech/dma/mini-project-nov/outputs/cleaned_data/order_products__prior.csv
C:/Users/KETAN SANJAY NIKAM/Desktop/mtech/dma/mini-project-nov/outputs/cleaned_data/products.csv
C:/Users/KETAN SANJAY NIKAM/Desktop/mtech/dma/mini-project-nov/outputs/cleaned_data/aisles.csv
C:/Users/KETAN SANJAY NIKAM/Desktop/mtech/dma/mini-project-nov/outputs/cleaned_data/departments.csv


SQL query

In [32]:
query = f"""
SELECT
    o.order_id,
    o.user_id,
    o.order_number,
    

    op.product_id,
    op.reordered,

    p.product_name,
    p.aisle_id,
    p.department_id

FROM read_csv_auto('{orders_path}') o

INNER JOIN read_csv_auto('{prior_path}') op
    ON o.order_id = op.order_id

INNER JOIN read_csv_auto('{products_path}') p
    ON op.product_id = p.product_id
"""

result = duckdb.query(query).df()

display(result.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

MemoryError: Unable to allocate 247. MiB for an array with shape (32434489,) and data type int64

In [27]:
con = duckdb.connect()

con.execute("SET memory_limit = '2GB' ")
con.execute("SET threads = 4 ")

print("DuckDB connection ready!")

DuckDB connection ready!


In [28]:
output_path = (
    INTEGRATED_DIR / "integrated_prior.parquet"
).resolve().as_posix()


con.execute(f"""
COPY (
    {query}
)
TO `{output_path}`
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Integration completed!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Integration completed!


In [29]:
output_file = INTEGRATED_DIR / "integrated_prior.parquet"

print("File exists:", output_file.exists())

if output_file.exists():
    print(
        "File size:",
        round(output_file.stat().st_size / (1024**3), 2),
        "GB"
    )

File exists: True
File size: 0.28 GB


In [30]:
preview = duckdb.query(f"""
    SELECT * FROM read_parquet(`{output_path}`)
    LIMIT 5
""").df()

display(preview)

,order_id,user_id,order_number,product_id,reordered,product_name,aisle_id,department_id
0,2,202279,3,33120,1,Organic Egg Whites,86,16
1,2,202279,3,28985,1,Michigan Organic Kale,83,4
2,2,202279,3,9327,0,Garlic Powder,104,13
3,2,202279,3,45918,1,Coconut Butter,19,13
4,2,202279,3,30035,0,Natural Sweetener,17,13


In [31]:
verification = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(DISTINCT user_id) AS unique_customers,
        COUNT(DISTINCT product_id) AS unique_products
    FROM read_parquet('{output_path}')
""").df()

display(verification)

,total_rows,unique_orders,unique_customers,unique_products
0,32434489,3214874,206209,49677


## Exploratory Data Analysis (EDA)